In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/uci-secom.csv')
print(df.shape)
df.info()

(1567, 592)
<class 'pandas.DataFrame'>
RangeIndex: 1567 entries, 0 to 1566
Columns: 592 entries, Time to Pass/Fail
dtypes: float64(590), int64(1), str(1)
memory usage: 7.1 MB


In [22]:
missing_ratio = df.isnull().mean().sort_values(ascending=False)
missing_ratio.head(20)

293    0.911934
292    0.911934
157    0.911934
158    0.911934
492    0.855775
220    0.855775
85     0.855775
358    0.855775
518    0.649649
382    0.649649
245    0.649649
244    0.649649
383    0.649649
384    0.649649
246    0.649649
517    0.649649
110    0.649649
109    0.649649
516    0.649649
111    0.649649
dtype: float64

In [23]:
df['Pass/Fail'].value_counts()

Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64

In [24]:
# 네 번째 셀 — 결측치 50% 넘는 컬럼 + Time 컬럼 제거:
threshold = 0.5
cols_to_drop = missing_ratio[missing_ratio > threshold].index
df_reduced = df.drop(columns=cols_to_drop).drop(columns=['Time'])
print(f"제거된 컬럼 수: {len(cols_to_drop)}")
print(f"남은 shape: {df_reduced.shape}")

제거된 컬럼 수: 28
남은 shape: (1567, 563)


In [25]:
# 다섯 번째 셀 — 피처(X)/라벨(y) 분리, 라벨을 0/1로 변환 (-1=정상→0, 1=불량→1):

X = df_reduced.drop(columns=['Pass/Fail'])
y = df_reduced['Pass/Fail'].map({-1: 0, 1: 1})
print(X.shape, y.value_counts())

(1567, 562) Pass/Fail
0    1463
1     104
Name: count, dtype: int64


In [26]:
#여섯 번째 셀 — 남은 결측치는 각 컬럼 중앙값으로 채우기:

X = X.fillna(X.median())
print("남은 결측치:", X.isnull().sum().sum())

남은 결측치: 0


In [27]:
#일곱 번째 셀 — 값이 전부 똑같은(정보 없는) 저분산 컬럼 제거:

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.0)
selector.fit(X)
X = X.loc[:, selector.get_support()]
print("저분산 컬럼 제거 후:", X.shape)

저분산 컬럼 제거 후: (1567, 446)


In [28]:
#여덟 번째 셀 — train/test 분리 (불균형 데이터라 stratify로 비율 유지):

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("train:", X_train.shape, y_train.value_counts().to_dict())
print("test:", X_test.shape, y_test.value_counts().to_dict())

train: (1253, 446) {0: 1170, 1: 83}
test: (314, 446) {0: 293, 1: 21}


In [29]:
#아홉 번째 셀 — SMOTE로 불량(소수 클래스) 데이터를 늘려주기

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print("SMOTE 적용 전:", y_train.value_counts().to_dict())
print("SMOTE 적용 후:", y_train_res.value_counts().to_dict())

SMOTE 적용 전: {0: 1170, 1: 83}
SMOTE 적용 후: {0: 1170, 1: 1170}


In [30]:
#열 번째 셀 — XGBoost 분류기 학습

from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    eval_metric='logloss'
)
model.fit(X_train_res, y_train_res)
print("학습 완료")

학습 완료


In [31]:
#열한 번째 셀 — test셋으로 성능 평가

from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['정상(Pass)', '불량(Fail)']))
print("혼동행렬 (행=실제, 열=예측):")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

    정상(Pass)       0.93      0.99      0.96       293
    불량(Fail)       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314

혼동행렬 (행=실제, 열=예측):
[[291   2]
 [ 21   0]]


In [32]:
# 열두 번째 셀 — 학습된 모델을 파일로 저장

import joblib
joblib.dump(model, '../app/model.pkl')
print("모델 저장 완료")

classification_report

모델 저장 완료


<function sklearn.metrics._classification.classification_report(y_true, y_pred, *, labels=None, target_names=None, sample_weight=None, digits=2, output_dict=False, zero_division='warn')>

In [33]:
##열한 번째 셀 — re / 좋은 소식이에요 — 제 쪽에서 똑같은 파이프라인으로 재현해봤는데 정확히 같은 현상이 나왔어요. 코드 실수가 아니라 이 데이터 자체가 어려운 데이터라서 생기는 거예요.
#확인해보니 AUC(모델이 정상/불량을 구분하는 전반적 능력)는 0.69로, 무작위(0.5)보다 뚜렷이 나은 신호가 있어요. 문제는 predict()가 기본으로 "불량 확률 50% 넘어야 불량 판정"인데, 실제 불량 21개 중 가장 높은 확률도 47.5%라 전부 문턱을 살짝 못 넘고 정상으로 분류된 거예요.
#이럴 땐 판정 기준(threshold)을 낮추는 게 일반적 해법이에요. 몇 가지로 실험해봤어요:
# threshold=0.3으로 변경
from sklearn.metrics import classification_report, confusion_matrix

threshold = 0.3
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= threshold).astype(int)

print(classification_report(y_test, y_pred, target_names=['정상(Pass)', '불량(Fail)']))
print("혼동행렬 (행=실제, 열=예측):")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

    정상(Pass)       0.94      0.97      0.96       293
    불량(Fail)       0.33      0.19      0.24        21

    accuracy                           0.92       314
   macro avg       0.64      0.58      0.60       314
weighted avg       0.90      0.92      0.91       314

혼동행렬 (행=실제, 열=예측):
[[285   8]
 [ 17   4]]


In [34]:
import joblib
joblib.dump({'model': model, 'threshold': threshold}, '../app/model.pkl')
print("모델 저장 완료")

모델 저장 완료


In [35]:
#열세 번째 셀 — 전체 test셋에 대한 SHAP 값 계산 (1~2분 정도 걸릴 수 있어요):

import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print(shap_values.shape)  # (샘플 수, 피처 수)

(314, 446)


In [36]:
# 열네 번째 셀 — 전체적으로 어떤 피처가 판정에 가장 큰 영향을 주는지(글로벌 중요도) 확인:

global_importance = pd.DataFrame({
    'feature': X_test.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

global_importance.head(15)

,feature,mean_abs_shap
389,511,0.361590
86,95,0.328721
31,33,0.326818
387,500,0.274832
320,419,0.257100
54,59,0.246321
375,486,0.238501
367,477,0.167036
203,247,0.152188
29,31,0.148544


In [37]:
# 열다섯 번째 셀 — 우리가 만들고 싶었던 핵심 기능: 특정 샘플 하나(웨이퍼 하나)에 대해 원인 파라미터 Top-5를 뽑는 함수. 대시보드에서 그대로 재사용할 거예요:

def explain_sample(position, top_n=5):
    row_shap = shap_values[position]
    ranking = pd.DataFrame({
        'feature': X_test.columns,
        'shap_value': row_shap
    })
    ranking['abs_shap'] = ranking['shap_value'].abs()
    return ranking.sort_values('abs_shap', ascending=False).head(top_n)[['feature', 'shap_value']]

# 모델이 불량으로 예측한 샘플 중 하나로 테스트
fail_positions = np.where(y_pred == 1)[0]
sample_pos = fail_positions[0]
print(f"샘플 위치: {sample_pos}, 실제 라벨: {y_test.iloc[sample_pos]}, 예측 불량 확률: {y_proba[sample_pos]:.3f}")
explain_sample(sample_pos)

샘플 위치: 42, 실제 라벨: 0, 예측 불량 확률: 0.353


,feature,shap_value
389,511,0.595958
367,477,-0.349875
421,561,-0.279398
118,131,-0.220606
317,416,-0.202106


In [38]:
# 열여섯 번째 셀 — 모델과 함께 대시보드에서 쓸 데이터도 저장:

import joblib

joblib.dump({
    'model': model,
    'threshold': threshold,
    'feature_names': list(X_test.columns)
}, '../app/model.pkl')

X_test.to_csv('../app/X_test_sample.csv', index=False)
y_test.to_csv('../app/y_test_sample.csv', index=False)
print("저장 완료")

저장 완료


SyntaxError: invalid syntax (3737097518.py, line 1)